In [1]:
### Full interactive NDVI dashboard - Harmonized combination l8-s2 for vegetation analysis 2013-2025

import os
import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Initialize Earth Engine
try:
    ee.Initialize()
    print("EE initialized")
except Exception:
    ee.Authenticate()
    ee.Initialize()
    print("EE authenticated and initialized")

## Setup / Load shapefile

bolivar_shp_path = r'C:\Users\Francesco\Python\earthlab\Ecuador\data\Bolivar\bolivar.shp'
bolivar_gdf = gpd.read_file(bolivar_shp_path).to_crs(epsg=4326)
bolivar = geemap.gdf_to_ee(bolivar_gdf)  # ee.Geometry/Feature for region

# 2. Cloud mask functions

def mask_landsat_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow_bit = 1 << 3
    snow_bit = 1 << 4
    cloud_bit = 1 << 5
    cirrus_bit = 1 << 7
    mask = (qa.bitwiseAnd(cloud_shadow_bit).eq(0)
            .And(qa.bitwiseAnd(snow_bit).eq(0))
            .And(qa.bitwiseAnd(cloud_bit).eq(0))
            .And(qa.bitwiseAnd(cirrus_bit).eq(0)))
    optical = image.select(['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']).multiply(0.0000275).add(-0.2)
    return optical.updateMask(mask).copyProperties(image, image.propertyNames())

def mask_sentinel2_clouds(image):
    # Uses the SCL (Scene Classification) band to mask clouds/shadows/snow
    scl = image.select('SCL')
    mask = (scl.neq(3)  # cloud shadow
            .And(scl.neq(8))   # cloud medium probability
            .And(scl.neq(9))   # cloud high probability
            .And(scl.neq(10))  # thin cirrus
            .And(scl.neq(11))) # snow/ice
    optical = image.select('B.*').divide(10000)
    out = optical.updateMask(mask).addBands(scl).copyProperties(image, image.propertyNames())
    return out

## Build combined collection

def combined_landsat_sentinel_ndvi(region):
    l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
          .filterBounds(region)
          .filterDate('2013-01-01', '2015-06-22')
          .map(mask_landsat_clouds)
          .map(lambda img: img.select(['SR_B5','SR_B4']).rename(['NIR','RED'])))
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(region)
          .filterDate('2015-06-23', '2025-12-31')
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 95))
          .map(mask_sentinel2_clouds)
          .map(lambda img: img.select(['B8','B4']).rename(['NIR','RED'])))

    combined = l8.merge(s2)
    return combined.map(lambda img: img.clip(region))

combined_ic = combined_landsat_sentinel_ndvi(bolivar)
print("Combined collection size:", combined_ic.size().getInfo())

## Yearly median NDVI collection

def yearly_median_ndvi(ic, start_year, end_year):
    years = ee.List.sequence(start_year, end_year)
    def per_year(y):
        y = ee.Number(y)
        start = ee.Date.fromYMD(y, 1, 1)
        end = ee.Date.fromYMD(y, 12, 31)
        coll = ic.filterDate(start, end)
        median_img = coll.median()
        ndvi = ee.Image(ee.Algorithms.If(coll.size().gt(0),
                                         median_img.normalizedDifference(['NIR','RED']).rename('NDVI'),
                                         ee.Image.constant(-9999).rename('NDVI').selfMask()))
        year_band = ee.Image.constant(y).rename('year').toFloat()
        return ndvi.addBands(year_band).set('year', y)
    return ee.ImageCollection(years.map(per_year))

yearly_ndvi_ic = yearly_median_ndvi(combined_ic, 2013, 2025)

## Keep images with at least one valid pixel

def has_data(img):
    count = img.select('NDVI').reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=bolivar.geometry(),
        scale=30,
        maxPixels=1e12
    ).get('NDVI')
    # updateMask using the condition so images with zero pixels become fully masked
    return ee.Image(img).updateMask(ee.Number(count).gt(0))

yearly_ndvi_ic_filtered = yearly_ndvi_ic.map(has_data)

## Trend and significance (server-side)

# Make sure the collection to reduce has 'year' and 'NDVI' bands
trend = yearly_ndvi_ic_filtered.select(['year','NDVI']).reduce(ee.Reducer.linearFit())
slope = trend.select('scale').clip(bolivar)
intercept = trend.select('offset').clip(bolivar)

pearson = yearly_ndvi_ic_filtered.select(['year','NDVI']).reduce(ee.Reducer.pearsonsCorrelation())
rho = pearson.select('correlation').clip(bolivar)
p_val = pearson.select('p_value').clip(bolivar)

# Decrease the spatial resolution to enable the visualization
#slope_display = slope.reduceResolution(reducer=ee.Reducer.mean(),
#                                       maxPixels=1024).reproject(crs='EPSG:4326',
#                                                                 scale=500).clip(bolivar)
#rho_display = rho.reduceResolution(reducer=ee.Reducer.mean(),
#                                   maxPixels=1024).reproject(crs='EPSG:4326',
#                                                             scale=500).clip(bolivar)

## Anomaly baseline (e.g., 2013-2018 baseline mean)

baseline_end = 2018
baseline_collection = yearly_ndvi_ic_filtered.filter(ee.Filter.lte('year', baseline_end))
baseline_mean = baseline_collection.select('NDVI').mean().rename('NDVI_baseline').clip(bolivar)

# create a dict of annual NDVI images (NDVI + year) for quick retrieval (optional)
# yearly_ndvi_ic_filtered is already available

## Build geemap UI

ndvi_vis = {'min': 0, 'max': 1, 'palette': ['brown','yellow','green']}
slope_vis = {'min': -0.01, 'max': 0.01, 'palette': ['red','yellow','green']}
rho_vis = {'min': -1, 'max': 1, 'palette': ['red','yellow','green']}
anomaly_vis = {'min': -0.5, 'max': 0.5, 'palette': ['blue','white','red']}  # NDVI anomaly

Map = geemap.Map(center=[-1.8, -79.1], zoom=9)
Map.add_basemap('HYBRID')
Map.addLayer(bolivar, {'color':'black'}, 'Bolivar boundary')

# Add slope and rho layers and capture layer objects
Map.addLayer(slope, slope_vis, 'NDVI slope (2013-2025)')
slope_layer = Map.layers[-1]
Map.addLayer(rho, rho_vis, 'Pearson rho (2013-2025)')
rho_layer = Map.layers[-1]

# initial opacities
slope_layer.opacity = 1
rho_layer.opacity = 0

# UI widgets
opacity_slider = widgets.FloatSlider(
    description='Fade (rho ↔ slope)',
    min=0, max=1, step=0.05, value=0,
    layout=widgets.Layout(width='420px')
)

year_slider = widgets.IntSlider(
    description='Year', value=2013, min=2013, max=2025, step=1,
    continuous_update=True, layout=widgets.Layout(width='420px')
)

show_anomaly_chk = widgets.Checkbox(value=False, description=f'Show anomaly vs {2013}-{baseline_end} mean')

# Output area for plots
ts_output = widgets.Output(layout={'border': '1px solid black'})

# Store current NDVI and anomaly layers to remove when updating
current_ndvi_layer = None
current_anomaly_layer = None
current_colorbar = None

# Helper: add NDVI layer for a selected year
def show_ndvi_for_year(year):
    global current_ndvi_layer, current_anomaly_layer, current_colorbar
    # remove previous
    if current_ndvi_layer is not None:
        Map.remove_layer(current_ndvi_layer)
        current_ndvi_layer = None
    if current_anomaly_layer is not None:
        Map.remove_layer(current_anomaly_layer)
        current_anomaly_layer = None
    if current_colorbar is not None:
        Map.remove_layer(current_colorbar)
        current_colorbar = None

    # get image for year
    img = ee.Image(yearly_ndvi_ic_filtered.filter(ee.Filter.eq('year', year)).first())
    Map.addLayer(img.select('NDVI'), ndvi_vis, f'NDVI {year}')
    current_ndvi_layer = Map.layers[-1]

    # optional anomaly layer
    if show_anomaly_chk.value:
        anomaly = img.select('NDVI').subtract(baseline_mean.select('NDVI_baseline')).rename('anomaly')
        Map.addLayer(anomaly, anomaly_vis, f'NDVI anomaly {year}')
        current_anomaly_layer = Map.layers[-1]

    # add colorbar for NDVI using branca
    current_colorbar = Map.add_colorbar_branca(ndvi_vis['palette'], vmin=ndvi_vis['min'], vmax=ndvi_vis['max'],
                                              caption=f'Median NDVI {year}')

# Opacity update
def on_opacity_change(change):
    a = change['new']
    slope_layer.opacity = 1 - a
    rho_layer.opacity = a

opacity_slider.observe(on_opacity_change, names='value')

# Year update
def on_year_change(change):
    year = change['new']
    show_ndvi_for_year(year)

year_slider.observe(on_year_change, names='value')
show_anomaly_chk.observe(lambda ch: show_ndvi_for_year(year_slider.value), names='value')

# Initial draw
show_ndvi_for_year(year_slider.value)

## Click inspector: sample NDVI for clicked point and plot time series

def sample_point_time_series(point):
    """
    point: ee.Geometry.Point
    returns: (years_list, ndvi_list) where ndvi_list contains floats or None
    """
    # Build a feature collection of year/ndvi for the filtered collection
    def to_feature(img):
        mean = img.select('NDVI').reduceRegion(ee.Reducer.mean(), point, 30, maxPixels=1e12).get('NDVI')
        year = img.get('year')
        return ee.Feature(None, {'year': year, 'NDVI': mean})
    fc = yearly_ndvi_ic_filtered.map(to_feature)
    # Get results (synchronous)
    years = fc.aggregate_array('year').getInfo()
    ndvis = fc.aggregate_array('NDVI').getInfo()
    # convert -9999 etc to None if present
    ndvis_clean = [None if (v is None or isinstance(v, float) and (str(v) == 'nan')) else v for v in ndvis]
    return years, ndvis_clean

# UI plotter
def plot_time_series_widget(years, ndvis, coords):
    ts_output.clear_output(wait=True)
    with ts_output:
        fig, ax = plt.subplots(figsize=(8,3.5))
        # convert to numeric lists and mask None
        years_num = [int(y) for y in years]
        ndvis_plot = [v if v is not None else float('nan') for v in ndvis]
        ax.plot(years_num, ndvis_plot, marker='o', linestyle='-')
        ax.set_ylim(-0.1,1.0)
        ax.set_xlim(min(years_num)-0.5, max(years_num)+0.5)
        ax.set_xlabel('Year')
        ax.set_ylabel('NDVI (median)')
        ax.set_title(f'NDVI time series at point ({coords[1]:.4f}, {coords[0]:.4f})')
        ax.grid(True, alpha=0.3)
        plt.show()

# Map click handler
def handle_map_interaction(**kwargs):
    # geemap passes e.g. {'type':'click','coordinates':[lon,lat], ...}
    if kwargs.get('type') == 'click':
        coords = kwargs.get('coordinates')
        lon, lat = coords[0], coords[1]
        point = ee.Geometry.Point(lon, lat)
        try:
            years, ndvis = sample_point_time_series(point)
            plot_time_series_widget(years, ndvis, coords)
        except Exception as e:
            ts_output.clear_output(wait=True)
            with ts_output:
                print("Error sampling point:", e)

# Attach handler
Map.on_interaction(handle_map_interaction)

## Layout and display

control_box = widgets.VBox([year_slider, show_anomaly_chk, opacity_slider])
left_box = widgets.VBox([Map, ts_output], layout=widgets.Layout(width='75%'))
ui = widgets.HBox([control_box, left_box], layout=widgets.Layout(align_items='flex-start'))

display(ui)
print("Click on the map to show the NDVI time series for that point.")


EE initialized


c:\Users\Francesco\anaconda3\envs\env\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files.  Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


Combined collection size: 1670


EEException: User memory limit exceeded.